# Setup

##  Imports

In [1]:
import os
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import torch
import torchaudio
import time
import openai

if not hasattr(torchaudio, 'set_audio_backend'):
    torchaudio.set_audio_backend = lambda x: None

from pyannote.audio import Pipeline
from pydub import AudioSegment

W0420 18:02:21.446000 33476 .venv\Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


## Environment & API Setup

In [2]:
load_dotenv()
api_key = os.getenv("AQUEDUCT_API_KEY", os.getenv("API_KEY"))
hf_token = os.getenv("HF_TOKEN")

if not api_key or not hf_token:
    raise ValueError("Missing API Keys. Ensure AQUEDUCT_API_KEY and HF_TOKEN are set in .env")

client = OpenAI(api_key=api_key, base_url="https://aqueduct.ai.datalab.tuwien.ac.at/v1")

## Local Model

In [3]:
print("Loading Pyannote Diarization Model locally...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
diarization_pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    token=hf_token
).to(device)

Loading Pyannote Diarization Model locally...


The torchaudio backend is switched to 'soundfile'. Note that 'sox_io' is not supported on Windows.
The torchaudio backend is switched to 'soundfile'. Note that 'sox_io' is not supported on Windows.


## Path Configuration

In [4]:
BASE_FOLDER = Path("D:/TUW-BT/user_study_recordings")

## Helper Functions

In [5]:
def transcribe_snippet(file_path: Path, max_retries: int = 5) -> str:
    """Sends audio to Aqueduct with rate-limit pacing and smart retries."""
    base_wait = 5  # Start waiting 5 seconds if the rate limit is hit

    for attempt in range(max_retries):
        try:
            with open(file_path, "rb") as audio_file:
                response = client.audio.transcriptions.create(
                    model="whisper-large",
                    file=audio_file
                )

            time.sleep(2.1) # Wait 2 seconds after successful call

            return response.text.strip()

        except openai.RateLimitError as e:
            # Double the wait time on each subsequent failure
            wait_time = base_wait * (2 ** attempt)
            print(f"  ⏳ Rate limit hit! Waiting {wait_time}s before retry {attempt + 1}/{max_retries}...")
            time.sleep(wait_time)

        except Exception as e:
            # Fallback check in case the API throws a generic exception with a 429 code
            if "429" in str(e):
                wait_time = base_wait * (2 ** attempt)
                print(f"  ⏳ Rate limit hit (429)! Waiting {wait_time}s before retry...")
                time.sleep(wait_time)
            else:
                print(f"❌ Error transcribing snippet: {e}")
                return "[Unintelligible / Transcription Error]"

    print(f"❌ Failed to transcribe {file_path.name} after maximum retries.")
    return "[Unintelligible / Transcription Error]"

def identify_speakers(transcripts: list, student_id: str) -> dict:
    speaker_map = {}
    instructor_id = None
    trigger_phrases = ["identification as instructor", "identify as instructor", "identification as an instructor"]

    for entry in transcripts:
        text_lower = entry["text"].lower()
        if any(phrase in text_lower for phrase in trigger_phrases):
            instructor_id = entry["speaker"]
            break

    if not instructor_id:
        print("  ⚠️ Warning: Trigger phrase not found. Defaulting SPEAKER_00 to Instructor.")
        instructor_id = "SPEAKER_00"

    for entry in transcripts:
        spk = entry["speaker"]
        if spk not in speaker_map:
            speaker_map[spk] = "Instructor" if spk == instructor_id else student_id

    return speaker_map

# Processing

In [6]:
for timeslot_folder in BASE_FOLDER.glob("Timeslot #*"):
    print(f"\n========================================")
    print(f"Processing: {timeslot_folder.name}")

    timeslot_num = timeslot_folder.name.split("#")[-1]
    student_name = f"Student #{timeslot_num}"

    audio_dir = timeslot_folder / "audio"
    sliced_dir = timeslot_folder / "sliced"
    transcript_dir = timeslot_folder / "transcript"

    sliced_dir.mkdir(parents=True, exist_ok=True)
    transcript_dir.mkdir(parents=True, exist_ok=True)

    audio_files = list(audio_dir.glob("*.*"))
    if not audio_files:
        print(f"  -> No audio file found in {audio_dir}. Skipping.")
        continue

    main_audio_path = audio_files[0]
    output_txt_path = transcript_dir / f"{timeslot_folder.name}.txt"

    if output_txt_path.exists():
        print(f"  -> Transcript already exists for this timeslot. Skipping.")
        continue

    print(f"  -> Loading audio into memory with Pydub...")
    full_audio = AudioSegment.from_file(str(main_audio_path))

    mono_audio = full_audio.set_channels(1)

    samples = np.array(mono_audio.get_array_of_samples())
    max_val = float(1 << (8 * mono_audio.sample_width - 1))
    samples = samples.astype(np.float32) / max_val
    waveform = torch.from_numpy(samples).unsqueeze(0)

    audio_in_memory = {
        "waveform": waveform,
        "sample_rate": mono_audio.frame_rate
    }

    print(f"  -> Running local diarization on {device}...")
    diarization = diarization_pipeline(audio_in_memory)

    print(f"  -> Slicing audio and transcribing via API...")
    chronological_transcripts = []
    snippet_idx = 0

    for turn, _, speaker in diarization.speaker_diarization.itertracks(yield_label=True):
        start_ms = int(turn.start * 1000)
        end_ms = int(turn.end * 1000)

        snippet = full_audio[start_ms:end_ms]

        snippet_filename = f"{snippet_idx:03d}_{speaker}.wav"
        snippet_path = sliced_dir / snippet_filename
        snippet.export(str(snippet_path), format="wav")

        text = transcribe_snippet(snippet_path)

        if text:
            chronological_transcripts.append({
                "speaker": speaker,
                "text": text,
                "start": turn.start
            })

        snippet_idx += 1

    print("  -> Mapping speakers...")
    speaker_mapping = identify_speakers(chronological_transcripts, student_name)

    print(f"  -> Compiling final transcript to {output_txt_path.name}...")
    with open(output_txt_path, "w", encoding="utf-8") as f:
        f.write(f"--- Transcript for {timeslot_folder.name} ---\n\n")
        current_speaker = None
        current_block = []

        for entry in chronological_transcripts:
            mapped_name = speaker_mapping.get(entry["speaker"], "Unknown Speaker")
            if mapped_name != current_speaker:
                if current_speaker:
                    f.write(f"{current_speaker}:\n{' '.join(current_block)}\n\n")
                current_speaker = mapped_name
                current_block = [entry["text"]]
            else:
                current_block.append(entry["text"])

        if current_speaker:
            f.write(f"{current_speaker}:\n{' '.join(current_block)}\n\n")

print("\n✅ All user study folders processed successfully!")


Processing: Timeslot #0
  -> Loading audio into memory with Pydub...
  -> Running local diarization on cuda...


C:\GitHub\Repositories\TUW-BT\.venv\Lib\site-packages\pyannote\audio\utils\reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(
C:\GitHub\Repositories\TUW-BT\.venv\Lib\site-packages\pyannote\audio\models\blocks\pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1858.)
  std = sequences.std(dim=-1, correction=1)


  -> Slicing audio and transcribing via API...
  -> Mapping speakers...
  ⚠️ Warning: Trigger phrase not found. Defaulting SPEAKER_00 to Instructor.
  -> Compiling final transcript to Timeslot #0.txt...

✅ All user study folders processed successfully!
